In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn import tree
from sklearn.model_selection import GridSearchCV


In [ ]:
# read data from repository
X_test = pd.read_csv('X_test.csv')
X_train = pd.read_csv('X_train.csv')
y_test = pd.read_csv('y_test.csv')
y_train = pd.read_csv('y_train.csv')

# compare y values in test and train sets
print(y_train.describe())
print(y_test.describe())

In [ ]:
# tree 1 without any constaints

tree_1 = tree.DecisionTreeClassifier(random_state= 12345)

tree_1.fit(X_train, y_train)

y_results = tree_1.predict(X_test)
y_results_proba = tree_1.predict_proba(X_test)

print(accuracy_score(y_test, y_results))

print(tree_1.get_depth())
print(tree_1.get_n_leaves())
# tree.plot_tree(tree_1)

# massive depth and number of leaves - pretty impossible to be plotted

In [ ]:
tree_2 = tree.DecisionTreeClassifier(max_depth=5, random_state=12345)

model = tree_2.fit(X_train, y_train)

y_results = tree_2.predict(X_test)
y_results_proba = tree_2.predict_proba(X_test)

print(accuracy_score(y_test, y_results))

print(tree_2.get_depth())
print(tree_2.get_n_leaves())
tree.plot_tree(tree_2, filled=True)


In [ ]:
# Cut the first tree using CCP_alpha

path = tree_1.cost_complexity_pruning_path(X_train, y_train)

ccp_alphas = path.ccp_alphas

# OPTIMIZATION: Take only meaningful alphas
meaningful_alphas = ccp_alphas[ccp_alphas > 0.0001]
print(len(meaningful_alphas))

trees = []

for ccp_alpha in meaningful_alphas:
    tree_cur = tree.DecisionTreeClassifier(random_state=12345, ccp_alpha=ccp_alpha)
    tree_cur.fit(X_train, y_train)
    trees.append(tree_cur)

print(len(trees))

In [ ]:
results = []

for tree_current in trees:
    prediction = tree_current.predict(X_test)
    result = accuracy_score(y_test, prediction) * 100
    results.append(result)

dep = []
leav = []
for tree_current in trees:
    depth = tree_current.get_depth()
    leave = tree_current.get_n_leaves()
    dep.append(depth)
    leav.append(leave)

print(dep)
print(leav)

best = 0.0000
best_num = 0
for i, percentage in enumerate(results):
    if percentage > best:
        best = percentage
        best_num = i

print(f"Best Accuracy on test:{best:.2f}%, iteration number: {best_num} - ccp_alpha: {meaningful_alphas[best_num]}, layers: {trees[best_num].get_depth()}")


In [ ]:
import matplotlib.pyplot as plt

best_tree = trees[best_num]

plt.figure(figsize=(20, 10))

tree.plot_tree(best_tree, max_depth=3, filled=True, fontsize=10)
plt.show()